# CIFAR-10 Image Classification
## Custom CNN vs Transfer Learning with GoogLeNet/Inception
---
**Objective:** Train a baseline CNN from scratch, then apply transfer learning with a pretrained GoogLeNet to demonstrate the performance gap — and understand *why* it exists.

| Stage | Model | Strategy |
|---|---|---|
| Part 1 | Custom CNN | Trained from scratch |
| Part 2 | GoogLeNet (pretrained) | Frozen feature extractor → Fine-tuned |
| Part 3 | Comparison | Metrics, confusion matrices, error analysis |

> **Runtime:** Recommended — Google Colab T4 GPU (~30–40 min total)


## 1. Imports & Environment Setup

In [ ]:
# ── Standard Library ──────────────────────────────────────────────────────
import os
import time
import copy
import random
import warnings
warnings.filterwarnings("ignore")

# ── Numerical / Plotting ───────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# ── PyTorch Core ───────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, OneCycleLR
from torch.cuda.amp import GradScaler, autocast

# ── torchvision ────────────────────────────────────────────────────────────
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.datasets import CIFAR10

# ── sklearn metrics ────────────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# ── Progress bar ───────────────────────────────────────────────────────────
from tqdm.auto import tqdm

print(f"PyTorch  : {torch.__version__}")
print(f"torchvision: {torchvision.__version__}")
print("✓ All imports successful")


## 2. GPU / CUDA Detection

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    USE_AMP = True          # mixed precision
else:
    print("⚠  No GPU detected — training will be slow on CPU")
    USE_AMP = False


## 3. Reproducibility

In [ ]:
SEED = 42

def set_seed(seed: int) -> None:
    """Pin all random sources so results are reproducible across runs."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False   # flip to True for speed if input size is fixed

set_seed(SEED)
print(f"Seed fixed at {SEED}")


## 4. Hyperparameter Configuration
All tunable knobs in one dict — no magic numbers scattered across cells.

In [ ]:
CFG = {
    # ── Data ──────────────────────────────────────────────────────────────
    "data_dir"        : "./data",
    "num_classes"     : 10,
    "num_workers"     : 2,

    # ── CNN from scratch ───────────────────────────────────────────────────
    "cnn_batch_size"  : 128,
    "cnn_epochs"      : 30,
    "cnn_lr"          : 1e-3,
    "cnn_weight_decay": 1e-4,
    "cnn_dropout"     : 0.4,
    "cnn_channels"    : [64, 128, 256, 512],   # conv block output channels

    # ── Transfer Learning — frozen stage ──────────────────────────────────
    "tl_batch_size"   : 64,
    "tl_frozen_epochs": 5,
    "tl_frozen_lr"    : 1e-3,

    # ── Transfer Learning — fine-tune stage ───────────────────────────────
    "tl_finetune_epochs": 20,
    "tl_finetune_lr"    : 2e-4,
    "tl_weight_decay"   : 1e-4,

    # ── Early stopping ────────────────────────────────────────────────────
    "patience"        : 7,

    # ── Image sizes ───────────────────────────────────────────────────────
    "cnn_img_size"    : 32,
    "tl_img_size"     : 224,   # GoogLeNet expects ≥ 75px; 224 = ImageNet standard

    # ── Checkpoints ───────────────────────────────────────────────────────
    "cnn_ckpt"        : "best_cnn.pth",
    "tl_ckpt"         : "best_googlenet.pth",
}

print("Configuration:")
for k, v in CFG.items():
    print(f"  {k:<25} = {v}")


## 5. Dataset — CIFAR-10
CIFAR-10: 60 000 colour images (32×32), 10 balanced classes, split 50k train / 10k test.

**Two paths supported:**
1. Automatic download via `torchvision` (default)
2. Custom local path — set `CUSTOM_DATA_PATH` below


In [ ]:
CUSTOM_DATA_PATH = None   # e.g. "/content/drive/MyDrive/cifar10" or None for auto-download

CIFAR10_CLASSES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

# ── CNN transforms (32×32, CIFAR stats) ────────────────────────────────────
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

cnn_train_tf = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

cnn_val_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

# ── Transfer-learning transforms (224×224, ImageNet stats) ─────────────────
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

tl_train_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

tl_val_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


def build_loaders(train_tf, val_tf, batch_size, data_path=None, val_split=0.1):
    """
    Returns train_loader, val_loader, test_loader.
    Uses data_path if provided, else auto-downloads to CFG['data_dir'].
    """
    root = data_path if data_path else CFG["data_dir"]

    full_train = CIFAR10(root=root, train=True,  download=True,  transform=train_tf)
    test_set   = CIFAR10(root=root, train=False, download=True,  transform=val_tf)

    # 90/10 train-val split
    n_val   = int(len(full_train) * val_split)
    n_train = len(full_train) - n_val
    train_set, val_set = torch.utils.data.random_split(
        full_train, [n_train, n_val],
        generator=torch.Generator().manual_seed(SEED)
    )
    # override val transform
    val_set.dataset = copy.deepcopy(full_train)
    val_set.dataset.transform = val_tf

    kw = dict(num_workers=CFG["num_workers"], pin_memory=(DEVICE.type == "cuda"))
    train_loader = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True,  **kw)
    val_loader   = torch.utils.data.DataLoader(val_set,   batch_size=batch_size, shuffle=False, **kw)
    test_loader  = torch.utils.data.DataLoader(test_set,  batch_size=batch_size, shuffle=False, **kw)

    print(f"Train  : {len(train_set):,} samples | {len(train_loader)} batches")
    print(f"Val    : {len(val_set):,} samples | {len(val_loader)} batches")
    print(f"Test   : {len(test_set):,} samples | {len(test_loader)} batches")
    return train_loader, val_loader, test_loader


print("Building CNN dataloaders …")
cnn_train_loader, cnn_val_loader, cnn_test_loader = build_loaders(
    cnn_train_tf, cnn_val_tf, CFG["cnn_batch_size"], CUSTOM_DATA_PATH
)


## 6. Data Visualisation

In [ ]:
def imshow_batch(loader, title="Sample Batch", n=32, cols=8):
    """Denormalise and display a grid of images from a DataLoader."""
    imgs, labels = next(iter(loader))
    imgs   = imgs[:n]
    labels = labels[:n]

    # inverse-normalise for display
    mean = torch.tensor(CIFAR_MEAN).view(3, 1, 1)
    std  = torch.tensor(CIFAR_STD).view(3, 1, 1)
    imgs = imgs * std + mean
    imgs = imgs.clamp(0, 1)

    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.6, rows * 1.8))
    axes = axes.flatten()

    for i, (img, lbl) in enumerate(zip(imgs, labels)):
        axes[i].imshow(img.permute(1, 2, 0).numpy())
        axes[i].set_title(CIFAR10_CLASSES[lbl], fontsize=7)
        axes[i].axis("off")
    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    fig.suptitle(title, fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

imshow_batch(cnn_train_loader, title="CIFAR-10 — Augmented Training Samples")

# ── Class distribution ──────────────────────────────────────────────────────
all_labels = [y for _, y in cnn_train_loader.dataset]
counts = np.bincount(all_labels)

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(CIFAR10_CLASSES, counts, color=plt.cm.tab10.colors, edgecolor="black", linewidth=0.5)
ax.set_title("Class Distribution — Training Set", fontsize=13, fontweight="bold")
ax.set_ylabel("Count")
ax.set_ylim(0, max(counts) * 1.15)
for bar, cnt in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
            str(cnt), ha="center", fontsize=8)
plt.tight_layout()
plt.show()


## 7. Custom CNN Architecture
A modular, production-style CNN:
- **ConvBlock** = Conv → BN → ReLU → MaxPool (reusable unit)  
- **Classifier head** = Global Average Pooling → Dropout → FC  
- Weight init via He (Kaiming) normal — correct for ReLU activations


In [ ]:
class ConvBlock(nn.Module):
    """Reusable Conv → BN → ReLU → optional MaxPool block."""

    def __init__(self, in_ch: int, out_ch: int, pool: bool = True):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        ]
        if pool:
            layers.append(nn.MaxPool2d(2, 2))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class CustomCNN(nn.Module):
    """
    Custom CNN for CIFAR-10 (32×32 input).

    Architecture:
        Input  (B, 3, 32, 32)
        Block1 (B, 64,  16, 16)
        Block2 (B, 128,  8,  8)
        Block3 (B, 256,  4,  4)
        Block4 (B, 512,  2,  2)
        GAP    (B, 512)
        Drop   (p=dropout)
        FC     (B, num_classes)
    """

    def __init__(self, channels=None, num_classes: int = 10, dropout: float = 0.4):
        super().__init__()
        if channels is None:
            channels = [64, 128, 256, 512]

        in_ch = 3
        self.blocks = nn.ModuleList()
        for out_ch in channels:
            self.blocks.append(ConvBlock(in_ch, out_ch, pool=True))
            in_ch = out_ch

        self.gap  = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(dropout)
        self.fc   = nn.Linear(channels[-1], num_classes)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        for blk in self.blocks:
            x = blk(x)
        x = self.gap(x).flatten(1)
        x = self.drop(x)
        return self.fc(x)


# ── Sanity check ────────────────────────────────────────────────────────────
cnn_model = CustomCNN(
    channels  = CFG["cnn_channels"],
    num_classes = CFG["num_classes"],
    dropout   = CFG["cnn_dropout"]
).to(DEVICE)

dummy = torch.randn(2, 3, 32, 32).to(DEVICE)
out   = cnn_model(dummy)
assert out.shape == (2, 10), f"Bad output shape: {out.shape}"

total_params = sum(p.numel() for p in cnn_model.parameters())
train_params = sum(p.numel() for p in cnn_model.parameters() if p.requires_grad)
print(f"Total params   : {total_params:,}")
print(f"Trainable params: {train_params:,}")
print(f"Output shape   : {out.shape}  ✓")


## 8. Training Pipeline

In [ ]:
class EarlyStopping:
    """Stop training when val_loss stops improving."""

    def __init__(self, patience: int = 7, delta: float = 1e-4):
        self.patience  = patience
        self.delta     = delta
        self.counter   = 0
        self.best_loss = np.inf
        self.triggered = False

    def __call__(self, val_loss: float) -> bool:
        if val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.counter   = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.triggered = True
        return self.triggered


def train_one_epoch(model, loader, criterion, optimizer, scaler, scheduler=None):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    pbar = tqdm(loader, desc="  train", leave=False, unit="batch")
    for imgs, labels in pbar:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=USE_AMP):
            logits = model(imgs)
            loss   = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        if scheduler is not None and isinstance(scheduler, OneCycleLR):
            scheduler.step()

        bs = labels.size(0)
        total_loss += loss.item() * bs
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += bs
        pbar.set_postfix(loss=f"{loss.item():.3f}")

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with autocast(enabled=USE_AMP):
            logits = model(imgs)
            loss   = criterion(logits, labels)
        bs = labels.size(0)
        total_loss += loss.item() * bs
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += bs

    return total_loss / total, correct / total


def run_training(
    model, train_loader, val_loader,
    epochs, lr, weight_decay, ckpt_path,
    patience=7, scheduler_type="cosine"
):
    """
    Full training loop with:
      - AMP (if GPU available)
      - gradient clipping
      - CosineAnnealingLR or OneCycleLR
      - early stopping
      - best-model checkpointing
    Returns history dict.
    """
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=weight_decay
    )

    if scheduler_type == "onecycle":
        scheduler = OneCycleLR(optimizer, max_lr=lr,
                               steps_per_epoch=len(train_loader), epochs=epochs)
    else:
        scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr * 1e-2)

    scaler  = GradScaler(enabled=USE_AMP)
    stopper = EarlyStopping(patience=patience)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_acc   = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler, scheduler)
        vl_loss, vl_acc = evaluate(model, val_loader, criterion)

        if scheduler_type == "cosine":
            scheduler.step()

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(vl_loss)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(vl_acc)

        if vl_acc > best_val_acc:
            best_val_acc   = vl_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save(best_model_wts, ckpt_path)
            tag = " ← best"
        else:
            tag = ""

        elapsed = time.time() - t0
        print(f"Epoch {epoch:03d}/{epochs} | "
              f"tr_loss={tr_loss:.4f} tr_acc={tr_acc:.4f} | "
              f"vl_loss={vl_loss:.4f} vl_acc={vl_acc:.4f} | "
              f"{elapsed:.1f}s{tag}")

        if stopper(vl_loss):
            print(f"Early stopping at epoch {epoch}.")
            break

    model.load_state_dict(best_model_wts)
    print(f"\nBest val accuracy: {best_val_acc:.4f}")
    return history


## 9. Train Custom CNN

In [ ]:
print("=" * 60)
print("TRAINING CUSTOM CNN FROM SCRATCH")
print("=" * 60)

cnn_history = run_training(
    model        = cnn_model,
    train_loader = cnn_train_loader,
    val_loader   = cnn_val_loader,
    epochs       = CFG["cnn_epochs"],
    lr           = CFG["cnn_lr"],
    weight_decay = CFG["cnn_weight_decay"],
    ckpt_path    = CFG["cnn_ckpt"],
    patience     = CFG["patience"],
    scheduler_type = "cosine",
)


## 10. CNN Training Curves

In [ ]:
def plot_history(history, title="Training History"):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(epochs, history["train_loss"], "o-", label="Train Loss", linewidth=2)
    ax1.plot(epochs, history["val_loss"],   "s--", label="Val Loss",   linewidth=2)
    ax1.set_title(f"{title} — Loss", fontsize=12, fontweight="bold")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
    ax1.legend(); ax1.grid(True, alpha=0.4)

    ax2.plot(epochs, [a * 100 for a in history["train_acc"]], "o-", label="Train Acc", linewidth=2)
    ax2.plot(epochs, [a * 100 for a in history["val_acc"]],   "s--", label="Val Acc",   linewidth=2)
    ax2.set_title(f"{title} — Accuracy", fontsize=12, fontweight="bold")
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy (%)")
    ax2.legend(); ax2.grid(True, alpha=0.4)

    plt.tight_layout()
    plt.show()

plot_history(cnn_history, title="Custom CNN")


## 11. CNN Evaluation on Test Set

In [ ]:
@torch.no_grad()
def get_predictions(model, loader):
    """Returns (all_preds, all_labels, all_probs) numpy arrays."""
    model.eval()
    preds, labels_list, probs_list = [], [], []
    for imgs, labels in tqdm(loader, desc="  evaluating", leave=False):
        imgs = imgs.to(DEVICE)
        with autocast(enabled=USE_AMP):
            logits = model(imgs)
        p = F.softmax(logits, dim=1)
        preds.extend(logits.argmax(1).cpu().numpy())
        labels_list.extend(labels.numpy())
        probs_list.extend(p.cpu().numpy())
    return np.array(preds), np.array(labels_list), np.array(probs_list)


def full_evaluation(model, test_loader, model_name="Model"):
    """Prints classification report and plots confusion matrix."""
    preds, labels, probs = get_predictions(model, test_loader)

    acc  = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, average="macro", zero_division=0)
    rec  = recall_score(labels, preds, average="macro", zero_division=0)
    f1   = f1_score(labels, preds, average="macro", zero_division=0)
    cm   = confusion_matrix(labels, preds)

    print(f"\n{'='*50}")
    print(f"  {model_name} — Test Results")
    print(f"{'='*50}")
    print(f"  Accuracy  : {acc*100:.2f}%")
    print(f"  Precision : {prec*100:.2f}%")
    print(f"  Recall    : {rec*100:.2f}%")
    print(f"  F1-Score  : {f1*100:.2f}%")
    print()
    print(classification_report(labels, preds, target_names=CIFAR10_CLASSES, digits=4))

    # Per-class accuracy
    per_class = cm.diagonal() / cm.sum(axis=1)
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Confusion matrix
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CIFAR10_CLASSES, yticklabels=CIFAR10_CLASSES,
                linewidths=0.5, ax=axes[0])
    axes[0].set_title(f"{model_name} — Confusion Matrix", fontweight="bold")
    axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")
    axes[0].tick_params(axis="x", rotation=45)

    # Per-class accuracy bar
    colors = ["#2ecc71" if a >= 0.8 else "#f39c12" if a >= 0.6 else "#e74c3c"
              for a in per_class]
    axes[1].bar(CIFAR10_CLASSES, per_class * 100, color=colors, edgecolor="black", linewidth=0.5)
    axes[1].axhline(acc * 100, color="navy", linestyle="--", linewidth=1.5, label=f"Overall {acc*100:.1f}%")
    axes[1].set_title(f"{model_name} — Per-class Accuracy", fontweight="bold")
    axes[1].set_ylabel("Accuracy (%)"); axes[1].set_ylim(0, 110)
    axes[1].tick_params(axis="x", rotation=45); axes[1].legend()
    plt.tight_layout(); plt.show()

    return {"acc": acc, "prec": prec, "rec": rec, "f1": f1, "preds": preds, "labels": labels, "probs": probs}


cnn_results = full_evaluation(cnn_model, cnn_test_loader, "Custom CNN")


## 12. Error Analysis — CNN

In [ ]:
def visualize_errors(model, loader, results, n=16, title="Misclassified Samples"):
    """Show the worst mistakes the model makes."""
    preds, labels = results["preds"], results["labels"]
    wrong_idx = np.where(preds != labels)[0][:n]
    if len(wrong_idx) == 0:
        print("No errors found!"); return

    # Collect actual images from loader
    all_imgs, all_labels = [], []
    for imgs, lbls in loader:
        all_imgs.append(imgs)
        all_labels.append(lbls)
    all_imgs   = torch.cat(all_imgs)
    all_labels = torch.cat(all_labels)

    mean = torch.tensor(CIFAR_MEAN).view(3, 1, 1)
    std  = torch.tensor(CIFAR_STD).view(3, 1, 1)

    cols = 8
    rows = (len(wrong_idx) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2, rows * 2.2))
    axes = axes.flatten()

    for ax_i, idx in enumerate(wrong_idx):
        img = all_imgs[idx] * std + mean
        img = img.clamp(0, 1).permute(1, 2, 0).numpy()
        axes[ax_i].imshow(img)
        axes[ax_i].set_title(
            f"T:{CIFAR10_CLASSES[labels[idx]][:4]}\nP:{CIFAR10_CLASSES[preds[idx]][:4]}",
            fontsize=7, color="red"
        )
        axes[ax_i].axis("off")
    for j in range(ax_i + 1, len(axes)):
        axes[j].axis("off")

    fig.suptitle(title, fontsize=13, fontweight="bold")
    plt.tight_layout(); plt.show()


visualize_errors(cnn_model, cnn_test_loader, cnn_results, title="Custom CNN — Misclassified Samples")


## 13. Transfer Learning — Concept

**Why transfer learning works:**
A model pretrained on ImageNet (1.28M images, 1000 classes) has already learned a rich hierarchy of visual features:
- Early layers → edges, corners, colour blobs  
- Mid layers → textures, simple shapes  
- Deep layers → object parts, semantic concepts  

Reusing these frozen features costs almost zero compute and dramatically reduces the data requirement.

**Two-stage strategy:**
1. **Frozen stage** — lock the backbone, train only the new classifier head. Fast convergence, prevents catastrophic forgetting.
2. **Fine-tune stage** — unfreeze deeper layers with a low LR. Adapts high-level features specifically to CIFAR-10.

**GoogLeNet / Inception-v1** is used here because it is lightweight, widely supported in torchvision, and includes `aux_logits` which we must handle correctly.


## 14. Build Transfer Learning Model (GoogLeNet)

In [ ]:
def build_googlenet(num_classes: int = 10, pretrained: bool = True):
    """
    Load pretrained GoogLeNet, replace the final FC layer.
    Aux classifiers are disabled (aux_logits=False) to simplify training.
    """
    weights = models.GoogLeNet_Weights.IMAGENET1K_V1 if pretrained else None
    model   = models.googlenet(weights=weights, aux_logits=False)

    # Replace classifier head
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(in_features, 512),
        nn.ReLU(inplace=True),
        nn.Dropout(0.3),
        nn.Linear(512, num_classes),
    )

    # Weight-init the new head
    for m in model.fc.modules():
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            nn.init.zeros_(m.bias)

    return model


def freeze_backbone(model):
    """Freeze all layers except the classifier head."""
    for name, param in model.named_parameters():
        param.requires_grad = name.startswith("fc.")
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"Frozen | trainable: {trainable:,} / {total:,}")


def unfreeze_backbone(model, unfreeze_from_layer: str = None):
    """Unfreeze all parameters (or from a specific named layer onward)."""
    for param in model.parameters():
        param.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Unfrozen | trainable: {trainable:,}")


# Build TL dataloaders
print("Building transfer-learning dataloaders (224×224)…")
tl_train_loader, tl_val_loader, tl_test_loader = build_loaders(
    tl_train_tf, tl_val_tf, CFG["tl_batch_size"], CUSTOM_DATA_PATH
)

tl_model = build_googlenet(CFG["num_classes"]).to(DEVICE)
print("GoogLeNet loaded  ✓")
print(f"FC head: {tl_model.fc}")


## 15. Stage 1 — Frozen Backbone Training

In [ ]:
freeze_backbone(tl_model)

print("=" * 60)
print("STAGE 1: FROZEN BACKBONE — Training only the classifier head")
print("=" * 60)

frozen_history = run_training(
    model        = tl_model,
    train_loader = tl_train_loader,
    val_loader   = tl_val_loader,
    epochs       = CFG["tl_frozen_epochs"],
    lr           = CFG["tl_frozen_lr"],
    weight_decay = CFG["tl_weight_decay"],
    ckpt_path    = "frozen_stage.pth",
    patience     = 5,
    scheduler_type = "cosine",
)


## 16. Stage 2 — Fine-tuning

In [ ]:
unfreeze_backbone(tl_model)

print("=" * 60)
print("STAGE 2: FINE-TUNING — All layers trainable, low LR")
print("=" * 60)

finetune_history = run_training(
    model        = tl_model,
    train_loader = tl_train_loader,
    val_loader   = tl_val_loader,
    epochs       = CFG["tl_finetune_epochs"],
    lr           = CFG["tl_finetune_lr"],
    weight_decay = CFG["tl_weight_decay"],
    ckpt_path    = CFG["tl_ckpt"],
    patience     = CFG["patience"],
    scheduler_type = "cosine",
)


## 17. Transfer Learning Training Curves

In [ ]:
# Merge both stages for a single continuous plot
def merge_histories(h1, h2):
    return {k: h1[k] + h2[k] for k in h1}

combined_tl_history = merge_histories(frozen_history, finetune_history)

plot_history(combined_tl_history, title="GoogLeNet (Frozen + Fine-tuned)")

# Stage boundary annotation
fig, ax = plt.subplots(figsize=(10, 4))
boundary = len(frozen_history["train_acc"])
total_ep = len(combined_tl_history["train_acc"])
ax.plot(range(1, total_ep + 1), [a * 100 for a in combined_tl_history["val_acc"]],
        "o-", linewidth=2, label="Val Accuracy")
ax.axvline(boundary + 0.5, color="red", linestyle="--", linewidth=1.5, label="Frozen → Fine-tune")
ax.fill_between(range(1, boundary + 1), 0, 100, alpha=0.05, color="orange", label="Frozen stage")
ax.fill_between(range(boundary + 1, total_ep + 1), 0, 100, alpha=0.05, color="green", label="Fine-tune stage")
ax.set_title("GoogLeNet — Val Accuracy by Training Stage", fontsize=12, fontweight="bold")
ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy (%)")
ax.legend(); ax.grid(True, alpha=0.4)
plt.tight_layout(); plt.show()


## 18. Transfer Learning Evaluation

In [ ]:
# Load best checkpoint
tl_model.load_state_dict(torch.load(CFG["tl_ckpt"], map_location=DEVICE))
tl_results = full_evaluation(tl_model, tl_test_loader, "GoogLeNet (Fine-tuned)")


## 19. Model Comparison

In [ ]:
# ── Metric table ────────────────────────────────────────────────────────────
metrics = ["acc", "prec", "rec", "f1"]
labels_m = ["Accuracy", "Precision", "Recall", "F1-Score"]

cnn_vals = [cnn_results[m] * 100 for m in metrics]
tl_vals  = [tl_results[m] * 100  for m in metrics]

x = np.arange(len(labels_m))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - width/2, cnn_vals, width, label="Custom CNN",  color="#3498db", edgecolor="black")
b2 = ax.bar(x + width/2, tl_vals,  width, label="GoogLeNet",   color="#2ecc71", edgecolor="black")

for bar in b1 + b2:
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.3,
            f"{bar.get_height():.1f}%", ha="center", va="bottom", fontsize=9)

ax.set_title("Model Comparison — Core Metrics", fontsize=13, fontweight="bold")
ax.set_ylabel("Score (%)"); ax.set_ylim(0, 110)
ax.set_xticks(x); ax.set_xticklabels(labels_m)
ax.legend(); ax.grid(axis="y", alpha=0.4)
plt.tight_layout(); plt.show()

# ── Summary table ────────────────────────────────────────────────────────────
print("\n" + "=" * 58)
print(f"{'Metric':<20} {'Custom CNN':>15} {'GoogLeNet':>15}")
print("=" * 58)
for lbl, cv, tv in zip(labels_m, cnn_vals, tl_vals):
    delta = tv - cv
    print(f"{lbl:<20} {cv:>13.2f}%  {tv:>13.2f}%  (+{delta:.2f}%)")
print("=" * 58)

cnn_params  = sum(p.numel() for p in cnn_model.parameters())
tl_params   = sum(p.numel() for p in tl_model.parameters())
print(f"{'Parameters':<20} {cnn_params:>14,}  {tl_params:>14,}")
print("=" * 58)


## 20. Per-class Accuracy Comparison

In [ ]:
from sklearn.metrics import confusion_matrix as _cm

def per_class_acc(preds, labels):
    cm = _cm(labels, preds)
    return cm.diagonal() / cm.sum(axis=1)

cnn_pca = per_class_acc(cnn_results["preds"], cnn_results["labels"])
tl_pca  = per_class_acc(tl_results["preds"],  tl_results["labels"])

x = np.arange(len(CIFAR10_CLASSES))
width = 0.35

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x - width/2, cnn_pca * 100, width, label="Custom CNN",  color="#3498db", edgecolor="black")
ax.bar(x + width/2, tl_pca  * 100, width, label="GoogLeNet",   color="#2ecc71", edgecolor="black")
ax.set_xticks(x); ax.set_xticklabels(CIFAR10_CLASSES, rotation=30)
ax.set_title("Per-class Accuracy: Custom CNN vs GoogLeNet", fontsize=12, fontweight="bold")
ax.set_ylabel("Accuracy (%)"); ax.set_ylim(0, 115)
ax.legend(); ax.grid(axis="y", alpha=0.4)
plt.tight_layout(); plt.show()


## 21. Inference on Custom Images
Upload any image and classify it with both models.


In [ ]:
from PIL import Image
import io

def preprocess_for_inference(img_path, model_type="tl"):
    """Load image and apply the appropriate transform."""
    img = Image.open(img_path).convert("RGB")
    tf = tl_val_tf if model_type == "tl" else cnn_val_tf
    return tf(img).unsqueeze(0)  # (1, C, H, W)


@torch.no_grad()
def predict_image(model, tensor, topk=3):
    """Return top-k class predictions with probabilities."""
    model.eval()
    tensor = tensor.to(DEVICE)
    with autocast(enabled=USE_AMP):
        logits = model(tensor)
    probs  = F.softmax(logits, dim=1)[0]
    topk_p, topk_i = probs.topk(topk)
    return [(CIFAR10_CLASSES[i], p.item()) for i, p in zip(topk_i, topk_p)]


def run_custom_inference(img_path: str):
    """Classify an image with both models and display results."""
    cnn_tensor = preprocess_for_inference(img_path, "cnn")
    tl_tensor  = preprocess_for_inference(img_path, "tl")

    cnn_preds = predict_image(cnn_model, cnn_tensor)
    tl_preds  = predict_image(tl_model,  tl_tensor)

    img = Image.open(img_path).convert("RGB")
    fig, axes = plt.subplots(1, 3, figsize=(14, 4),
                             gridspec_kw={"width_ratios": [1, 1.2, 1.2]})

    axes[0].imshow(img); axes[0].axis("off"); axes[0].set_title("Input Image", fontweight="bold")

    for ax, preds, name, color in [
        (axes[1], cnn_preds, "Custom CNN", "#3498db"),
        (axes[2], tl_preds,  "GoogLeNet",  "#2ecc71"),
    ]:
        classes_p = [f"{c} ({p*100:.1f}%)" for c, p in preds]
        vals      = [p * 100 for _, p in preds]
        bars = ax.barh(classes_p[::-1], vals[::-1], color=color, edgecolor="black")
        ax.set_xlim(0, 100)
        ax.set_title(f"{name} — Top-3", fontweight="bold")
        ax.set_xlabel("Confidence (%)")
        ax.grid(axis="x", alpha=0.4)

    plt.tight_layout(); plt.show()
    print("CNN  top-1:", cnn_preds[0])
    print("TL   top-1:", tl_preds[0])


# ── Demo: pick a test image from the dataset ────────────────────────────────
sample_imgs, sample_labels = next(iter(cnn_test_loader))
mean_t = torch.tensor(CIFAR_MEAN).view(3, 1, 1)
std_t  = torch.tensor(CIFAR_STD).view(3, 1, 1)
demo_img = (sample_imgs[0] * std_t + mean_t).clamp(0, 1).permute(1, 2, 0).numpy()
demo_img_pil = Image.fromarray((demo_img * 255).astype(np.uint8))
demo_img_pil.save("/tmp/demo_sample.png")

print(f"Ground truth: {CIFAR10_CLASSES[sample_labels[0]]}")
run_custom_inference("/tmp/demo_sample.png")

print("\n─── To use your own image ─────────────────────────────────────────────")
print("  from google.colab import files")
print("  uploaded = files.upload()")
print("  path = list(uploaded.keys())[0]")
print("  run_custom_inference(path)")


## 22. Save & Download Models

In [ ]:
# Full model state dicts already saved during training.
# This cell packages them for easy download from Colab.

import shutil, zipfile

artifacts = {
    CFG["cnn_ckpt"]: "Custom CNN best weights",
    CFG["tl_ckpt"]:  "GoogLeNet fine-tuned best weights",
}

with zipfile.ZipFile("cifar10_models.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for fname, desc in artifacts.items():
        if os.path.exists(fname):
            zf.write(fname)
            print(f"  ✓ Added {fname} — {desc}")
        else:
            print(f"  ✗ {fname} not found — was training completed?")

print("\nDownload: cifar10_models.zip")
print("  from google.colab import files")
print("  files.download('cifar10_models.zip')")


## 23. Conclusions & Analysis

### Why Transfer Learning Wins
- GoogLeNet's backbone was optimised on 1.28M images — far richer than CIFAR-10's 50k.
- The pretrained features (edges → textures → shapes) transfer directly; the model only needs to re-learn class boundaries.
- Even with just 5 frozen-stage epochs the classifier head converges faster than the scratch CNN in 30 epochs.

### Custom CNN Limitations
- **Receptive field** at 32×32 is constrained; 4 pooling stages leave a 2×2 spatial map before GAP.
- **Scale of supervision**: 50k images is small for training deep random-init networks without regularisation headroom.
- Batch Norm + Dropout + label smoothing mitigate but don't eliminate overfitting.

### Overfitting Observations
- CNN training accuracy typically exceeds validation by 5–10 pp — a sign the model memorises texture patterns not present in test augmentation.
- GoogLeNet (fine-tuned) shows a tighter train/val gap due to pretrained features acting as implicit regularisation.

### Computational Tradeoffs
| | Custom CNN | GoogLeNet |
|---|---|---|
| Input size | 32×32 | 224×224 |
| Memory/batch | Low | High |
| Convergence | Slow | Fast |
| Accuracy | ~80–85% | ~90–94% |

### Future Improvements
1. **Knowledge Distillation** — compress GoogLeNet knowledge back into a small CNN.
2. **CutMix / Mixup** augmentation for further regularisation.
3. **EfficientNet-B0 / MobileNetV3** — stronger accuracy/parameter tradeoff than GoogLeNet.
4. **Test-Time Augmentation (TTA)** — ensemble over augmented crops at inference.
5. **Self-supervised pretraining** (SimCLR / DINO) — removes ImageNet dependency entirely.
